In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install gensim
import gensim
import re # Regular Expression Library
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer
from gensim.parsing.preprocessing import remove_stopwords
from nltk.tokenize import word_tokenize # Tokenizaion
from spacy.lang.en import English
from spacy.lang.en.stop_words import STOP_WORDS

In [ ]:
# Plotting libraries
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# sklearn :
import sklearn
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier # Import Decision Tree Classifier
from sklearn.model_selection import train_test_split # Import train_test_split function
from sklearn import metrics #Import scikit-learn metrics module for accuracy calculation
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_curve, auc

In [ ]:
# load the dataset
# print the dataset
df = pd.read_csv('Review.csv')
df.head(100)

In [ ]:
# diaplay the column name of our dataset
df.columns

In [ ]:
# information about the data set
df.info()
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# Row an column in the dataset
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
df = df.drop([ 'Reviewer', 'Date', 'Title'], axis=1)

In [ ]:
df.head(100)

In [ ]:
# Check for duplicate rows in the entire dataset
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

# Display a few duplicates if they exist
if duplicate_count > 0:
    display(df[df.duplicated()].head())

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()
print(f"Duplicates removed. New dataset shape: {df.shape}")

In [ ]:
plt.figure(figsize=(15, 8))
sns.countplot(x='Company', data=df, palette='viridis')
plt.title('Distribution of Sentiments by Company')
plt.xlabel('Company')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
num_companies = df['Company'].nunique()
print(f"There are {num_companies} unique companies in the dataset.")

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
def convert_sentiment(rating):
    if rating <= 3:
        return 0  # Negative
    elif rating == 4:
        return 1  # Neutral
    else:
        return 2  # Positive

df['Sentiment'] = df['Rating'].apply(convert_sentiment)
print("Target distribution:")
display(df['Sentiment'].value_counts())
display(df.head())

In [ ]:
import matplotlib.pyplot as plt

# Count values
sentiment_counts = df['Sentiment'].value_counts()

# Plot
plt.figure()
sentiment_counts.plot(kind='bar')

plt.title("Sentiment Distribution")
plt.xlabel("Sentiment (0=Neg, 1=Neu, 2=Pos)")
plt.ylabel("Count")

plt.show()

In [ ]:
# 3. Data Cleaning
df = df[['Review', 'Rating']].dropna()

* Removal of Mentions:

In social media, Mentions are used to call/mention another user into our post. Generally, mentions don't have an added value to our model. So we will remove them.

A mention has a special pattern: **@UserName,** So we will remove all string which starts with @

In [ ]:
# Removal of Mentions:

## Creating a fucntion that will be applied to our datset :
def RemoveMentions(Review):
    text_ = re.sub(r"@\S+", "", Review)
    return text_


## Applying the function to each row of the data
print("=========== Before Removing Mentions ============\n")
print("\t" + df.loc[5, "Review"])
print("\n=========== After Removing Mentions ===========\n")
df["Review"] = df["Review"].apply(RemoveMentions)
print("\t" + df.loc[5, "Review"])

In [ ]:
# Defining a list containing punctuation signs of english :
punctuations_list = string.punctuation


## Defining that will be applied to our datset :
def RemovePunctuations(Review):
    transformator = str.maketrans('', '', punctuations_list)
    return Review.translate(transformator)


## Applying the fucntion to all rows :
print("=========== Before Removing Punctuations =============\n")
print("\t" + df.loc[10, "Review"])
print("\n=========== After Removing Punctuations \===========\n")
df["Review"] = df["Review"].apply(RemovePunctuations)
print("\t" + df.loc[10, "Review"])

In [ ]:
df.loc[10, "Review"]

In [ ]:
# Getting the pre defined stop words from nltk library :
stopwords = stopwords.words('english')

## Copying the df to use other libraries (spacy and gensim)
df_copy1 = df.loc[:100].copy(deep=True)
df_copy2 = df.copy(deep=True)  # deep copy to create another df

## Applying the fucntion to all rows
print("=========== Before Removing Stop words ============\n")
print("\t" + df_copy2.loc[12, "Review"])
print("\n=========== After Removing Stop words ===========\n")

## Exclude stopwords with Python's list comprehension and pandas.DataFrame.apply.
df_copy2['Review'] = df_copy2['Review'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stopwords)]))
print("\t" + df_copy2.loc[12, "Review"])

In [ ]:
df.loc[5, "Review"]

In [ ]:
## Creating a fucntion that will be applied to our datset :
def RemoveStopsSpacy(Review):
    # Load English tokenizer, tagger, parser, NER and word vectors
    nlp = English()

    #  "nlp" Object is used to create documents with linguistic annotations.
    my_doc = nlp(Review)

    # Create list of word tokens
    token_list = []
    for token in my_doc:
        token_list.append(token.text)
    # Create list of word tokens after removing stopwords
    filtered_sentence = []

    for word in token_list:
        lexeme = nlp.vocab[word]
        if lexeme.is_stop == False:
            filtered_sentence.append(word)
    return filtered_sentence


## Applying the fucntion to all rows
print("=========== Before Removing Stop words with spaCy ===========\n")
print("\t" + df_copy1.loc[12, "Review"])
print("\n=========== After Removing Stop words with spaCy ===========\n")

## Exclude stopwords with Python's list comprehension and pandas.DataFrame.apply.
df_copy1['Review'] = df_copy1['Review'].apply(lambda x: ' '.join(RemoveStopsSpacy(x)))
print("\t" + df_copy1.loc[12, "Review"])

In [ ]:
df.loc[8]

In [ ]:
## Applying the fucntion to all rows
print("=========== Before Removing Stop words with Gensim =======\n")
print("\t" + df.loc[12, "Review"])
print("\n=========== After Removing Stop words with Gensim =======\n")
df['Review'] = df['Review'].apply(lambda x: gensim.parsing.preprocessing.remove_stopwords(x))
print("\t" + df.loc[12, "Review"])

In [ ]:
## Creating a fucntion that will be applied to our datset :
def RemoveLinks(Review):
    return re.sub(r"http\S+", "", Review)

## Applying the fucntion to all rows of our dataset :
print("====== Before Removing Hyperlinks =====\n")
print("\t" + df.loc[0, "Review"])  # let's see for example the first row, which contains an hyperlink.
print("\n====== After Removing Hyperlinks ====\n")
df['Review'] = df['Review'].apply(RemoveLinks)
print("\t" + df.loc[0, "Review"])

In [ ]:
## Creating a fucntion that will be applied to our datset :
def RemoveNumbers(Review):
    return re.sub(r"[0-9]+", "", Review)

## Applying the fucntion to all rows
print("=========== Before Removing Numbers =======\n")
print("\t" + df.loc[2,"Review"])  #let's see for example the thirs row, which contains an number 50
print("\n=========== After Removing Numbers ========\n")
df['Review'] = df['Review'].apply(RemoveNumbers)
print("\t" + df.loc[2,"Review"])

In [ ]:
## Creating a fucntion that will be applied to our datset :
def RemoveWhitespaces(Review):
    Review = Review.strip()  # Leading and trailing whitespaces are removed
    return re.sub(r" +"," ",Review)

## Applying the fucntion to all rows :
df['Review'] = df['Review'].apply(lambda x: RemoveWhitespaces(x))

In [ ]:
# And now, let's see our tweet content feature:
print("The number of unique values of the text feature is {}".format(df['Review'].nunique()))
print("The total number of rows in our dataframe is : {}".format(len(df)))
print("The number of duplicated rows in our dataframe is : {}".format(len(df)-df['Review'].nunique()))

In [ ]:
# Removing duplicate row records but keeping original text : ( we only keep the first duplicate )
df = df.drop_duplicates(subset='Review', keep='first')

In [ ]:
# Checking if duplicates have been removed:
print("The number of unique values of the text feature is {}".format(df['Review'].nunique()))
print("The total number of rows in our dataframe is : {}".format(len(df)))
print("The number of duplicated rows in our dataframe is : {}".format(len(df)-df['Review'].nunique()))

In [ ]:
# NLTK (Natural Language Toolkit) provides a utility function for tokenizing data.
df['tokenized_Review'] = df['Review'].apply(word_tokenize)

# Remove the old 'tokenized_tweets' column if it exists to clean up the output
if 'tokenized_tweets' in df.columns:
    df.drop(columns=['tokenized_tweets'], inplace=True)

df.head()

In [ ]:
# Creating an instance of the stemmer :
stemmer = PorterStemmer()

## Creating a fucntion that will be applied to our datset :
def Stemmer(Review):
    return " ".join([stemmer.stem(word) for word in Review])

## Applying the fucntion to all rows :
df['tokenized_Review_stemmed'] = df['tokenized_Review'].apply(lambda Review: Stemmer(Review))

In [ ]:
import nltk
nltk.download('wordnet')

# Creating an instance of the limmatizer :
wordnet_lemmatizer = WordNetLemmatizer()

# Applying the limmatizer to all rows:
df['tokenized_Review_stemmed_lemmatized'] = df['tokenized_Review_stemmed'].apply(
    lambda text: wordnet_lemmatizer.lemmatize(text, pos="v"))

In [ ]:
df.head(50)

In [ ]:
from google.colab import files

# Save the preprocessed dataframe to a CSV
output_filename = 'preprocessed_car_reviews.csv'
df.to_csv(output_filename, index=False)

# Trigger the download to your local machine
files.download(output_filename)

In [ ]:
# Let's create a function which creates a wordcloud of a given pandas Series object :
def wordCloud(data_pos, max_words):
    # call the wordcloud function to show the most top 1000 used words:
    cloud = WordCloud(max_words=max_words, background_color="white", width=1600, height=800,
                      collocations=False).generate(" ".join(data_pos))
    plt.figure(figsize=(20, 20))
    plt.imshow(cloud)

In [ ]:
df['text_length'] = df['Review'].apply(len)

plt.hist(df['text_length'], bins=30)
plt.title("Text Length Distribution")
plt.show()

In [ ]:
sns.pairplot(df)
plt.show()

In [ ]:
def convert_sentiment(rating):
    if rating <= 3:
        return 0  # Negative
    elif rating == 4:
        return 1  # Neutral
    else:
        return 2  # Positive

df['Sentiment'] = df['Rating'].apply(convert_sentiment)
wordCloud(df.loc[df["Sentiment"] == 1, "Review"],2000)

In [ ]:
# Get the counts of each sentiment
sentiment_counts = df['Sentiment'].value_counts()

# Map sentiment labels for better readability on the plot
sentiment_labels = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
sentiment_counts.index = sentiment_counts.index.map(sentiment_labels)

# Create a bar chart
plt.figure(figsize=(8, 6))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette='viridis')
plt.title('Distribution of Sentiments (Negative, Neutral, Positive)')
plt.xlabel('Sentiment Category')
plt.ylabel('Number of Reviews')
plt.show()

In [ ]:
!pip install imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. We first need to convert text to numbers (Vectorization) before SMOTE
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['tokenized_Review_stemmed_lemmatized'])
y = df['Sentiment']

# 2. Apply SMOTE to balance the dataset
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_tfidf, y)

print(f'Original dataset shape: {y.value_counts().to_dict()}')
print(f'Resampled dataset shape: {y_res.value_counts().to_dict()}')

# 3. New Train-Test Split with balanced data
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)

In [ ]:
# Seperating input feature and label
X = df['tokenized_Review_stemmed_lemmatized']
y = df['Sentiment']


In [ ]:
# 6. Train-Test Split
# We use the processed text column 'tokenized_Review_stemmed_lemmatized' as X
X_train, X_test, y_train, y_test = train_test_split(
    df['tokenized_Review_stemmed_lemmatized'],
    df['Sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=df['Sentiment']
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

In [ ]:
def model_evaluation(model, X_train, X_test, y_train, y_test):
    # Train the model
    y_pred = model.predict(X_test)

    # Print the evalution metrics for the dataset

    print(classification_report(y_test, y_pred))

    # compute and plot the Confusion matrix
    cf_matrix = confusion_matrix(y_test, y_pred)
    categories = ['Negative', 'Neutral', 'Positive']
    group_names = ['True Neg', 'False Pos', 'False Neg', 'True Pos']
    group_percentages = ['{0:.2%}'.format(value) for value in cf_matrix.flatten() / np.sum(cf_matrix)]
    labels = [f'{v1}\n{v2}' for v1, v2 in zip(group_names,group_persecentages)]
    labels = np.asarray(labels).reshape(2,2)
    sns.heatmap(cf_matrix, annot = labels, cmap = 'Blues',fmt = '',
                xticklabels = categories, yticklabels = categories)
    plt.xlabel("Predicted values", fontdict = {'size':14}, labelpad = 10)
    plt.ylabel("Actual values" , fontdict = {'size':14}, labelpad = 10)
    plt.title ("Confusion Matrix", fontdict = {'size':18}, pad = 20)


In [ ]:
import math
int(math.sqrt(len(df))) # k = sqrt(n)

In [ ]:
# model-4 : K nearest neighbors using balanced data
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

start1 = time.time()
knn.fit(X_train_bal, y_train_bal)
end1 = time.time()
print("Training time: {:.2f}s".format(end1 - start1))

start2 = time.time()
y_pred4 = knn.predict(X_test_bal)
end2 = time.time()
print("Testing time: {:.2f}s".format(end2 - start2))
print("KNN Classification Report:")
print(classification_report(y_test_bal, y_pred4))

In [ ]:
from sklearn.preprocessing import label_binarize

# 1. Binarize labels for the balanced test set
y_test_bin = label_binarize(y_test_bal, classes=[0, 1, 2])
n_classes = y_test_bin.shape[1]

# 2. Get prediction probabilities
y_score4 = knn.predict_proba(X_test_bal)

# 3. Compute ROC curve and AUC for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score4[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# 4. Plotting
plt.figure(figsize=(8, 6))
colors = ['red', 'blue', 'green']
labels = ['Negative', 'Neutral','Positive']

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(labels[i], roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multiclass ROC Curve - KNN')
plt.legend(loc="lower right")
plt.show()